## Optimize CNMF

In [ ]:
import os
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import mesmerize_core as mc
import mesmerize_viz
import fastplotlib as fpl

from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

from pathlib import Path

if os.name == "nt":
    # disable the cache on windows, this will be automatic in a future version
    cnmf_cache.set_maxsize(0)

pd.options.display.max_colwidth = 120

## Set paths

In [ ]:
# data path
data_path = Path('D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001')
print(f"Load data from {data_path}.")

# Set output path and assign it as raw data path (required by df.caiman.add_item below)
export_path = Path('H:/Analysis/2P/C57_O1M2/10022023/run1/mesmerize/')
mc.set_parent_raw_data_path(export_path) 

# Create export_path if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)

# movie path
movie_path = Path.joinpath(export_path, 'cat_tiff_bt.tiff')

# batch path
batch_path = Path.joinpath(export_path, 'batch.pickle')

print(f"Export data to {export_path}.") 
print(f"Movie path: {movie_path}.") 
print(f"Batch path: {batch_path}.") 

### Load a batch

In [ ]:
df = mc.load_batch(batch_path)
df

### Cleanup DataFrame
We do this here, instead of at the end of the motion correction step, because calling `remove_item()` on windows will raise a `PermissionError` if the memmap file open. There is currently no way to close a `numpy.memmap`.

In [ ]:
# make a list of rows we want to keep using the uuids
rows_keep = [df.iloc[0].uuid, df.iloc[1].uuid, df.iloc[2].uuid]
rows_keep

In [ ]:
for i, row in df.iterrows():
    if row.uuid not in rows_keep:
         df.caiman.remove_item(row.uuid)
df

Indices are always reset when you use `caiman.remove_item()`. UUIDs are always preserved.

In [ ]:
# If a clipped memmap file exists, delete the original memmap file and rename the clipped memmap file to the original name
 
outputs_dict = df.iloc[0]["outputs"]
mc_memmapped_fname = Path.joinpath(export_path, outputs_dict['mcorr-output-path'])
print(f"Original memmap file: {mc_memmapped_fname}.")

# Check if a clipped memmap file exists (i.e. same name as original but with "clipped_" appended to the beginning)
clipped_mmap_path = Path.joinpath(mc_memmapped_fname.parent,
                                  f"clipped_{mc_memmapped_fname.name}")
# check if the clipped memmap file exists
if clipped_mmap_path.exists():
    print(f"Clipped memmap file exists: {clipped_mmap_path}.")
    # if it does, delete the original memmap file
    mc_memmapped_fname.unlink()
    # and rename the clipped memmap file to the original name
    clipped_mmap_path.rename(mc_memmapped_fname)
    print(f"Clipped memmap file renamed to {mc_memmapped_fname}.")

### CNMF

Perform CNMF using the mcorr output.

Similar to mcorr, put the CNMF params within the `main` key. The `refit` key will perform a second iteration, as shown in the `caiman` `demo_pipeline.ipynb` notebook.

In [ ]:
# some params for CNMF
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 20, # framerate, very important!
            'p': 1,
            'nb': 1,
            'merge_thr': 0.80,
            'rf': 20,
            'stride': 10, # "stride" for cnmf, "strides" for mcorr
            'K': 8,
            'gSig': [5, 5],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.0,
            'rval_thr': 0.7,
            'use_cnn': True,
            'min_cnn_thr': 0.8,
            'cnn_lowest': 0.1,
            'decay_time': 0.4,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

### Add CNMF item

You can provide the mcorr item row to `input_movie_path` and it will resolve the path of the input movie from the entry in the DataFrame.

In [ ]:
good_mcorr_index = 2 # 0
 
# add a batch item
df.caiman.add_item(
    algo='cnmf', # algo is cnmf
    input_movie_path=df.iloc[good_mcorr_index],  # use mcorr output from a completed batch item
    params=params_cnmf,
    item_name=df.iloc[good_mcorr_index]["item_name"], # use the same item name
)

See the cnmf item at the bottom of the dataframe

In [ ]:
df

### Run CNMF

The API is identical to running mcorr

In [ ]:
index = -1  # most recently added item
df.iloc[index].caiman.run()

### Reload dataframe

In [ ]:
df = df.caiman.reload_from_disk()
df

### CNMF outputs

Similar to mcorr, you can use the `mesmerize-core` API to fetch the outputs. The API reference for CNMF is here: https://mesmerize-core.readthedocs.io/en/latest/api/cnmf.html

In [ ]:
index = -1  # index of the cnmf item, last item in the dataframe

# temporal components
temporal = df.iloc[index].cnmf.get_temporal()

In [ ]:
temporal.shape

Many of the cnmf functions take a rich set of arguments

In [ ]:
# get accepted or rejected components
temporal_good = df.iloc[index].cnmf.get_temporal("good")

# shape is [n_components, n_frames]
temporal_good.shape

In [ ]:
# get specific components
df.iloc[index].cnmf.get_temporal(np.array([1, 5, 9]))

In [ ]:
# get temporal with the residuals, i.e. C + YrA
temporal_with_residuals = df.iloc[index].cnmf.get_temporal(add_residuals=True)

In [ ]:
# get contours
contours = df.iloc[index].cnmf.get_contours()

Returns: `(list of np.ndarray of contour coordinates, list of center of mass)`

In [ ]:
print(f"contour 0 coordinates:\n\n{contours[0][0]}\n\n com: {contours[1][0]}")

In [ ]:
len(contours)

In [ ]:
# get_contours() also takes arguments
contours_good = df.iloc[index].cnmf.get_contours("good")

In [ ]:
len(contours_good[0]) # number of contours

swap_dim

In [ ]:
# get the first contour using swap_dim=True (default)
swap_dim_true = df.iloc[index].cnmf.get_contours()[0][0]

In [ ]:
# get the first contour  with swap_dim=False
swap_dim_false = df.iloc[index].cnmf.get_contours(swap_dim=False)[0][0]

In [ ]:
plt.plot(
    swap_dim_true[:, 0], 
    swap_dim_true[:, 1], 
    label="swap_dim=True"
)
plt.plot(
    swap_dim_false[:, 0], 
    swap_dim_false[:, 1], 
    label="swap_dim=False"
)
plt.legend()

In [ ]:
# swap_dim swaps the x and y dims
plt.plot(
    swap_dim_true[:, 0], 
    swap_dim_true[:, 1], 
    linewidth=30
)
plt.plot(
    swap_dim_false[:, 1], 
    swap_dim_false[:, 0], 
    linewidth=10
)

### Reconstructed movie - `A * C`
### Reconstructed background - `b * f`
### Residuals - `Y - AC - bf` 

Mesmerize-core provides these outputs as lazy arrays. This allows you to work with arrays that would otherwise be hundreds of gigabytes or terabytes in size.

In [ ]:
rcm = df.iloc[index].cnmf.get_rcm()
rcm

LazyArrays behave like numpy arrays

In [ ]:
rcm[42]

In [ ]:
rcm[10:20].shape

### Using LazyArrays

In [ ]:
rcm_accepted = df.iloc[index].cnmf.get_rcm("good")
rcm_rejected = df.iloc[index].cnmf.get_rcm("bad")
corr_image = df.iloc[index].caiman.get_corr_image()

In [ ]:
%gui qt

In [ ]:
iw_max = fpl.ImageWidget(
    data=[corr_image, rcm_accepted.max_image, rcm_rejected.max_image],
    names=["corr_image", "accepted", "rejected"],
    grid_shape = (1, 3),
    grid_plot_kwargs={"size": (1200, 800)},
    cmap="gray"
)
iw_max.show()

In [ ]:
iw_max.close()

In [ ]:
iw_rcm_separated = fpl.ImageWidget(
    data=[rcm_accepted, rcm_rejected],
    names=["accepted", "rejected"],
    grid_plot_kwargs={"size": (900, 450)},
    cmap="gray"
)
iw_rcm_separated.show()

### All CNMF LazyArrays

In [ ]:
rcb = df.iloc[index].cnmf.get_rcb()
residuals = df.iloc[index].cnmf.get_residuals()
input_movie = df.iloc[index].caiman.get_input_movie()

`ImageWidget` accepts arrays that are sufficiently numpy-like

In [ ]:
iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

In [ ]:
iw_rcm.close()

### Visualize everything with `mesmerize-viz`

In [ ]:
# viz_cnmf = df.cnmf.viz()
# viz_cnmf.show()

In [ ]:
# viz_cnmf.close()

#### Run component evaluation

In [ ]:
index = -1

In [ ]:
cnmf_obj = df.iloc[index].cnmf.get_output()
print(cnmf_obj.estimates.SNR_comp)

In [ ]:
# Get inf values
SNR_vals = cnmf_obj.estimates.SNR_comp
inf_val_idx = np.where(np.isinf(SNR_vals))
# Print indices and values
print(inf_val_idx)
print(SNR_vals[inf_val_idx])
# Change inf values to 0
# SNR_vals[np.isinf(SNR_vals)] = 0
# print(SNR_vals[inf_val_idx])

#### Kushal's hack to replace inf values with the max values from the array and save the cnmf obj. (untested)

In [ ]:
# import cache
from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

# SNR_vals is cnmf_obj.estimates.SNR_comp
SNR_vals[np.isinf(SNR_vals)] = np.nanmax(SNR_vals[~np.isinf(SNR_vals)])

# get path of cnmf obj on disk so we can overwrite
path = df.iloc[index].cnmf.get_output_path()

# save cnmf obj to disk
cnmf_obj.save(path)

# clear cache
cnmf_cache.clear_cache()

# viz should work now

In [ ]:
# df.iloc[1].cnmf.run_eval({'SNR_lowest': 0.5, 'min_SNR': 2})

In [ ]:
# df.iloc[index].cnmf.get_output_path()

In [ ]:
%gui qt

In [ ]:
viz_simple = df.cnmf.viz(
    # image_data_options=["input", "rcm"],
    image_data_options=["max"],
)
# viz_simple.show(sidecar=True)
viz_simple.show()

In [ ]:
viz_simple.close()

### More customization of kwargs

In [ ]:
# viz_more_custom = df.cnmf.viz(
#     image_data_options=["input", "rcm", "rcm-max", "corr"],
#     temporal_kwargs={"add_residuals": True},
# )

In [ ]:
# viz_more_custom.show(sidecar=True)

### Parameter Gridsearch

As shown for motion correction, the purpose of `mesmerize-core` is to perform parameter searches

In [ ]:
# itertools.product makes it easy to loop through parameter variants
# basically, it's easier to read that deeply nested for loops
from itertools import product

# variants of several parameters
gSig_variants = [4, 6]
K_variants = [4, 8]
merge_thr_variants = [0.8, 0.95]

# always use deepcopy like before
new_params_cnmf = deepcopy(params_cnmf)

# create a parameter grid
parameter_grid = product(gSig_variants, K_variants, merge_thr_variants)

# a single for loop to go through all the various parameter combinations
for gSig, K, merge_thr in parameter_grid:
    # deep copy params dict just like before
    new_params_cnmf = deepcopy(new_params_cnmf)
    
    new_params_cnmf["main"]["gSig"] = [gSig, gSig]
    new_params_cnmf["main"]["K"] = K
    new_params_cnmf["main"]["merge_thr"] = merge_thr
    
    # add param combination variant to batch
    df.caiman.add_item(
        algo="cnmf",
        item_name=df.iloc[good_mcorr_index]["item_name"],  # good mcorr item
        input_movie_path=df.iloc[good_mcorr_index],
        params=new_params_cnmf
    )

We now have lot of cnmf items

In [ ]:
df

View the diffs

In [ ]:
df.caiman.get_params_diffs(algo="cnmf", item_name=df.iloc[-1]["item_name"])

### Run the `cnmf` batch items

In [ ]:
for i, row in df.iterrows():
    if row["outputs"] is not None: # item has already been run
        continue # skip
        
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

### Load outputs

In [ ]:
df = df.caiman.reload_from_disk()

In [ ]:
df

### Visualize with `mesmerize-viz`

In [ ]:
viz_cnmf = df.cnmf.viz()
viz_cnmf.show(sidecar=True)

### Caiman docs on component eval

https://caiman.readthedocs.io/en/latest/Getting_Started.html#component-evaluation

> The quality of detected components is evaluated with three parameters:
>
> Spatial footprint consistency (rval): The spatial footprint of the component is compared with the frames where this component is active. Other component’s signals are subtracted from these frames, and the resulting raw data is correlated against the spatial component. This ensures that the raw data at the spatial footprint aligns with the extracted trace.
>
> Trace signal-noise-ratio (SNR): Peak SNR is calculated from strong calcium transients and the noise estimate.
>
> CNN-based classifier (cnn): The shape of components is evaluated by a 4-layered convolutional neural network trained on a manually annotated dataset. The CNN assigns a value of 0-1 to each component depending on its resemblance to a neuronal soma.
> 
> Each parameter has a low threshold:
> - (rval_lowest (default -1), SNR_lowest (default 0.5), cnn_lowest (default 0.1))
>
> and high threshold
> 
> - (rval_thr (default 0.8), min_SNR (default 2.5), min_cnn_thr (default 0.9))
> 
> A component has to exceed ALL low thresholds as well as ONE high threshold to be accepted.

### This rich visualization is still customizable!

Public attributes:

- `image_widget`: the `ImageWidget` in the visualization
- `plot_temporal`: the `Plot` with the temporal
- `plot_heatmap`: the `Plot` with the heatmap
- `cnmf_obj`: The cnmf object currently being visualized. This object gets saved to disk when you click the "Save Eval to disk" button.
- `component_index`: current component index, `int`

A few public methods:
- `show()` show the visualization
- `set_component_index(index: int)` manually set the component index

In [ ]:
viz_cnmf.image_widget.cmap = "gray"

In [ ]:
viz_cnmf.plot_heatmap

In [ ]:
viz_cnmf.plot_heatmap["heatmap"].cmap = "viridis"

In [ ]:
viz_cnmf.plot_heatmap["heatmap"].cmap.vmax

In [ ]:
viz_cnmf.plot_heatmap["heatmap"].cmap.vmax = 2_000

Customize contours

In [ ]:
for subplot in viz_cnmf.image_widget.gridplot:
    subplot["contours"][:].thickness = 1.0

In [ ]:
for subplot in viz_cnmf.image_widget.gridplot:
    subplot["contours"].visible = False

In [ ]:
for subplot in viz_cnmf.image_widget.gridplot:
    subplot["contours"].visible = True

In [ ]:
viz_cnmf.plot_temporal["line"].thickness()

In [ ]:
viz_cnmf.plot_temporal["line"].thickness = 1

### Visualize fewer things

In [ ]:
viz_simple = df.cnmf.viz(
    # image_data_options=["input", "rcm"],
    image_data_options=["input"],
)
viz_simple.show(sidecar=True)

In [ ]:
viz_simple.close()

### More customization of kwargs

In [ ]:
viz_more_custom = df.cnmf.viz(
    image_data_options=["input", "rcm", "rcm-max", "corr"],
    temporal_kwargs={"add_residuals": True},
)

In [ ]:
viz_more_custom.show(sidecar=True)